# Sentence-level Emotion Analysis (RoBERTa GoEmotions)

This notebook computes emotion scores **per sentence** for each essay, averages them to obtain **essay-level emotion profiles**, and then compares the **average emotion differences** between AI-generated and humanized essays.


In [ ]:
import pandas as pd
import numpy as np
from transformers import pipeline
from tqdm import tqdm
import re

# Display all columns when inspecting dataframes
pd.set_option('display.max_columns', None)

print("Loading RoBERTa GoEmotions model (SamLowe/roberta-base-go_emotions)...")
emotion_classifier = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=None  # get scores for all emotions
)

print("Loading dataset...")
df = pd.read_csv("../datasets.csv")
print(df.shape)
print(df.columns.tolist())

In [ ]:
def split_into_sentences(text: str):
    """Simple sentence splitter using punctuation. Adjust if you need more accuracy."""
    if pd.isna(text) or text.strip() == "":
        return []
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text.strip())
    # Split on ., !, ? followed by space or end of string
    sentences = re.split(r"(?<=[.!?])\s+", text)
    # Drop very short fragments
    sentences = [s.strip() for s in sentences if len(s.strip()) > 0]
    return sentences


def get_emotion_scores(text: str, max_chars: int = 2000):
    """Run the GoEmotions classifier on a single text and return {emotion: score} dict."""
    if pd.isna(text) or text.strip() == "":
        return None

    # Truncate long text for safety
    if len(text) > max_chars:
        text = text[:max_chars]

    try:
        results = emotion_classifier(text)
        # results is a list with one element; that element is a list of {label, score}
        emotion_dict = {item["label"]: item["score"] for item in results[0]}
        return emotion_dict
    except Exception as e:
        print(f"Error processing text: {e}")
        return None


def get_sentence_level_emotions(text: str):
    """Return per-sentence emotion scores and essay-level averages.

    Returns
    -------
    sentences: list[str]
    sentence_scores: list[dict]  # one dict per sentence
    essay_avg: dict              # mean score per emotion across sentences
    """
    sentences = split_into_sentences(text)
    sentence_scores = []

    for sent in sentences:
        scores = get_emotion_scores(sent)
        if scores is not None:
            sentence_scores.append(scores)

    if not sentence_scores:
        return sentences, [], None

    # Convert list of dicts to DataFrame and average across sentences
    sent_df = pd.DataFrame(sentence_scores)
    essay_avg = sent_df.mean(axis=0).to_dict()
    return sentences, sentence_scores, essay_avg

In [ ]:
# Compute sentence-level emotions for AI and humanized essays

ai_sentences = []              # list[list[str]] per essay
ai_sentence_scores = []        # list[list[dict]] per essay
ai_essay_avg_emotions = []     # list[dict] per essay

human_sentences = []
human_sentence_scores = []
human_essay_avg_emotions = []

print("Processing AI essays sentence by sentence...")
for text in tqdm(df["AI_essay"].values, desc="AI essays"):
    sents, s_scores, avg_scores = get_sentence_level_emotions(text)
    ai_sentences.append(sents)
    ai_sentence_scores.append(s_scores)
    ai_essay_avg_emotions.append(avg_scores)

print("Processing humanized essays sentence by sentence...")
for text in tqdm(df["humanized_essay"].values, desc="Humanized essays"):
    sents, s_scores, avg_scores = get_sentence_level_emotions(text)
    human_sentences.append(sents)
    human_sentence_scores.append(s_scores)
    human_essay_avg_emotions.append(avg_scores)

# Attach to dataframe for reference / further analysis
df["AI_sentences"] = ai_sentences
df["AI_sentence_emotions"] = ai_sentence_scores
df["AI_essay_avg_emotions"] = ai_essay_avg_emotions

df["human_sentences"] = human_sentences
df["human_sentence_emotions"] = human_sentence_scores
df["human_essay_avg_emotions"] = human_essay_avg_emotions

# Optionally save an intermediate file with sentence-level info
df.to_csv("../datasets_sentence_level_emotions.csv", index=False)
print("Saved ../datasets_sentence_level_emotions.csv")

In [ ]:
# Build a flat table of essay-level averaged emotion scores (from sentence-level outputs)

# Get full list of emotions from the first non-empty essay
all_emotions = None
for avg in ai_essay_avg_emotions:
    if isinstance(avg, dict) and avg:
        all_emotions = list(avg.keys())
        break

if all_emotions is None:
    raise ValueError("No emotion scores were computed; check the pipeline and dataset.")

# Create columns for each emotion based on sentence-level averages
for emotion in all_emotions:
    df[f"AI_sentavg_{emotion}"] = [
        (avg.get(emotion) if isinstance(avg, dict) and avg is not None else np.nan)
        for avg in ai_essay_avg_emotions
    ]
    df[f"human_sentavg_{emotion}"] = [
        (avg.get(emotion) if isinstance(avg, dict) and avg is not None else np.nan)
        for avg in human_essay_avg_emotions
    ]

# Compute overall mean (across essays) of these sentence-averaged scores
ai_avg_emotions = {
    emotion: df[f"AI_sentavg_{emotion}"].mean() for emotion in all_emotions
}
human_avg_emotions = {
    emotion: df[f"human_sentavg_{emotion}"].mean() for emotion in all_emotions
}

comparison = pd.DataFrame({
    "Emotion": all_emotions,
    "AI_SentenceAvg": [ai_avg_emotions[e] for e in all_emotions],
    "Humanized_SentenceAvg": [human_avg_emotions[e] for e in all_emotions],
})
comparison["Difference_Humanized_minus_AI"] = (
    comparison["Humanized_SentenceAvg"] - comparison["AI_SentenceAvg"]
)
comparison = comparison.sort_values("Difference_Humanized_minus_AI", ascending=False)

comparison.head()

In [ ]:
# Inspect the top and bottom emotions by sentence-level averaged difference

print("Top 10 emotions where humanized essays have higher average (sentence-level) scores:")
print(comparison.head(10).to_string(index=False))

print("\nTop 10 emotions where AI essays have higher average (sentence-level) scores:")
print(comparison.tail(10).sort_values("Difference_Humanized_minus_AI").to_string(index=False))

# Save results for later analysis/comparison with the previous whole-essay method
comparison.to_csv("../emotion_comparison_sentence_level.csv", index=False)
print("\nSaved ../emotion_comparison_sentence_level.csv")